# Test segmentation + stim-mask generation on EXP_25

Sanity-check the full pipeline (raw TIFF → cellpose → `StimSpotOnCell` mask) against
an old Moritz EXP_25 acquisition. The per-FOV stim location (top/middle/bottom) is
looked up from `exp25.parquet`, so each FOV gets the same treatment it received
originally.

Raw TIFFs live on the share — multi-channel `(C, H, W)`, channel 0 = mRuby2 / ERK-KTR
(same channel Moritz used for segmentation in `Moritz_DMD_top-bottom-side-spot_cellpose_EXP_24.ipynb`).

In [1]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt

from faro.segmentation.cellpose import SegmentorCellpose
from faro.stimulation.spot_on_cell import StimSpotOnCell

RAW_DIR = r"\\izbkingston.izb.unibe.ch\imaging.data\PertzLab\mkwasny\Experiments\Opto_stim\exp_25\raw"
PARQUET = "exp25.parquet"
TEST_FRAME = 5            # which timepoint to load (matches STIM_FRAME in the analysis)
SEG_CHANNEL = 0           # mRuby2 / ERK-KTR (same as Moritz's supertracker)

# Reuse the exact stimulator + segmentor from point_stimulation.ipynb so this test
# also validates that those constructors are behaving sensibly.
segmentor  = SegmentorCellpose(model="cyto3", diameter=60, flow_threshold=1.0,
                                cellprob_threshold=-1.0, min_size=100)
stimulator = StimSpotOnCell(spot_radius=5, height_percentage=0.6, offset=-5,
                             clip_to_cell=True)

# FOV -> stim_location mapping from the parquet.
_meta = (pd.read_parquet(PARQUET)[["fov", "stim_location", "treatment"]]
          .drop_duplicates("fov").set_index("fov"))

def process_fov(fov: int, frame: int = TEST_FRAME):
    """Load TIFF -> segment -> stim mask. Returns (raw_image, labels, stim_mask, stim_location)."""
    fname = f"{fov:03d}_{frame:05d}.tiff"
    raw = tifffile.imread(os.path.join(RAW_DIR, fname))     # (C, H, W) uint16
    labels = segmentor.segment(raw[SEG_CHANNEL])
    stim_location = _meta.loc[fov, "stim_location"]
    stim_mask, _ = stimulator.get_stim_mask(
        {"labels": labels}, metadata={"stim_location": stim_location}
    )
    return raw, labels, stim_mask, stim_location

ModuleNotFoundError: No module named 'faro.stimulation.spot_on_cell'

In [ ]:
# Pick one example FOV per stim location.
example_fovs = (_meta.reset_index().drop_duplicates("stim_location")
                .set_index("stim_location")["fov"].to_dict())
print("example FOVs:", example_fovs)

fig, axes = plt.subplots(len(example_fovs), 3, figsize=(12, 4 * len(example_fovs)), dpi=120)
for row, (loc, fov) in enumerate(example_fovs.items()):
    raw, labels, stim_mask, expected_loc = process_fov(int(fov))
    ax_raw, ax_lbl, ax_stim = axes[row]

    ax_raw.imshow(raw[SEG_CHANNEL], cmap="gray")
    ax_raw.set_title(f"FOV {fov} — raw ch{SEG_CHANNEL} ({_meta.loc[fov,'treatment']})")

    ax_lbl.imshow(labels % 23, cmap="tab20b")
    ax_lbl.set_title(f"cellpose labels ({labels.max()} cells)")

    ax_stim.imshow(raw[SEG_CHANNEL], cmap="gray")
    ax_stim.contour(labels > 0, levels=[0.5], colors="cyan", linewidths=0.4)
    ax_stim.imshow(np.where(stim_mask, 1, np.nan), cmap="autumn", alpha=0.9)
    ax_stim.set_title(f"stim mask — location={expected_loc}")

    for ax in axes[row]:
        ax.set_axis_off()

plt.tight_layout()
plt.show()